# Iterative Fine-Tuning Notebook

This notebook implements iterative fine-tuning on labeled cell patches from the Indian satellite dataset. Each run:
1. Loads the latest checkpoint (.pth)
2. Trains on accumulated labeled cells
3. Tracks validation metrics
4. Saves updated weights with metadata

Use this for multi-round refinement: label cells → train → evaluate → label more.


## Section 1: Import Libraries and Configure Environment

In [ ]:
import os
import sys
import json
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from tqdm import tqdm

# Add scripts to path for imports
sys.path.insert(0, str(Path.cwd()))

# Set reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Configure device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Import from codebase
from scripts.landcover_pipeline import CANONICAL_CLASS_NAMES, get_device, load_model


## Section 2: Load Existing .pth Checkpoint

Load the latest checkpoint and inspect its structure.


In [ ]:
# Configuration
CHECKPOINT_PATH = 'best_eurosat_model.pth'  # Change to 'best_indian_model.pth' if available
OUTPUT_DIR = Path('training_history')
OUTPUT_DIR.mkdir(exist_ok=True)

# Try to load checkpoint; if it's a dictionary with 'model_state', extract it
print(f"Loading checkpoint: {CHECKPOINT_PATH}")
checkpoint_full = torch.load(CHECKPOINT_PATH, map_location=device)

# Handle checkpoint format: could be direct state_dict or wrapped in dict
if isinstance(checkpoint_full, dict) and 'model_state_dict' in checkpoint_full:
    checkpoint_data = checkpoint_full
    model_state = checkpoint_full['model_state_dict']
    print(f"Checkpoint is wrapped with keys: {checkpoint_full.keys()}")
    print(f"  - Epoch: {checkpoint_full.get('epoch', 'N/A')}")
    print(f"  - Best loss: {checkpoint_full.get('best_loss', 'N/A')}")
else:
    # Direct state_dict
    checkpoint_data = None
    model_state = checkpoint_full
    print("Checkpoint is a direct state_dict")

print(f"\nModel state keys (first 5): {list(model_state.keys())[:5]}")
print(f"Total parameters in state: {len(model_state)}")


## Section 3: Prepare Fine-Tuning Dataset

Load labeled cell patches from `data/cell_training/labeled_cells/` and create data loaders.


In [ ]:
class CellDataset(Dataset):
    """Load cell patches from class folders."""
    
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []
        self.class_names = CANONICAL_CLASS_NAMES
        self.class_to_idx = {name: idx for idx, name in enumerate(self.class_names)}
        
        # Collect all images from class folders
        for class_name in self.class_names:
            class_dir = self.root_dir / class_name
            if class_dir.exists():
                images = list(class_dir.glob('*.png')) + list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.tif'))
                for img_path in images:
                    self.samples.append((img_path, self.class_to_idx[class_name]))
        
        print(f"Dataset: {len(self.samples)} samples from {len(self.class_names)} classes")
        for class_name in self.class_names:
            count = sum(1 for _, label in self.samples if label == self.class_to_idx[class_name])
            print(f"  {class_name}: {count}")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

# Data transforms
train_transform = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Load dataset
data_dir = Path('data/cell_training/labeled_cells')
if not data_dir.exists():
    print(f"⚠️  Dataset directory not found: {data_dir}")
    print("Ensure you've extracted cell patches using 'extract_corrected_cells.py'")
else:
    full_dataset = CellDataset(data_dir, transform=None)
    
    # 80/20 train/val split
    val_size = max(1, int(len(full_dataset) * 0.2))
    train_size = len(full_dataset) - val_size
    
    train_dataset, val_dataset = torch.utils.data.random_split(
        full_dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED)
    )
    
    # Apply transforms to datasets
    train_dataset.dataset.transform = train_transform
    val_dataset.dataset.transform = val_transform
    
    print(f"\nDataset split:")
    print(f"  Training: {len(train_dataset)}")
    print(f"  Validation: {len(val_dataset)}")


## Section 4: Define Model and Restore Weights

Create the model architecture and load pretrained weights.


In [ ]:
# Build model architecture (Wide ResNet-50-2)
model = models.wide_resnet50_2(pretrained=False)

# Replace classification head for 9 classes + LogSoftmax
num_classes = len(CANONICAL_CLASS_NAMES)
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(in_features, num_classes),
    nn.LogSoftmax(dim=1)
)

model = model.to(device)

# Load checkpoint weights into model
print(f"Loading weights from checkpoint...")
try:
    if isinstance(checkpoint_full, dict) and 'model_state_dict' in checkpoint_full:
        model.load_state_dict(checkpoint_full['model_state_dict'])
    else:
        model.load_state_dict(checkpoint_full)
    print("✓ Weights loaded successfully")
except Exception as e:
    print(f"⚠️  Could not load weights: {e}")
    print("Starting from scratch")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel: Wide ResNet-50-2")
print(f"  Total params: {total_params:,}")
print(f"  Trainable params: {trainable_params:,}")
print(f"  Classes: {num_classes} {CANONICAL_CLASS_NAMES}")


## Section 5: Set Up Loss, Optimizer, and Scheduler

Configure loss function, optimizer, and learning rate scheduler.


In [ ]:
# Fine-tuning parameters
LEARNING_RATE = 1e-5
BATCH_SIZE = 32
EPOCHS = 15
WEIGHT_DECAY = 1e-4
PATIENCE = 5  # Early stopping patience

# Loss and optimizer
criterion = nn.NLLLoss()  # Since model outputs LogSoftmax
optimizer = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2,
    verbose=True
)

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Fine-tuning configuration:")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print(f"  Early stopping patience: {PATIENCE}")
print(f"  Optimizer: Adam")
print(f"  Loss: NLLLoss (for LogSoftmax output)")


## Section 6: Run Refinement Training Loop

Train the model for multiple epochs with early stopping.


In [ ]:
# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'epochs': []
}

best_val_loss = float('inf')
patience_counter = 0
start_time = datetime.now()

print(f"\n{'='*70}")
print(f"Starting Fine-Tuning | {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*70}\n")

for epoch in range(EPOCHS):
    # Training phase
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_correct += predicted.eq(labels).sum().item()
        train_total += labels.size(0)
        
        pbar.set_postfix({'loss': loss.item():.4f})
    
    # Validation phase (section 7)
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()
            val_total += labels.size(0)
    
    # Calculate metrics
    train_loss /= len(train_loader)
    train_acc = 100 * train_correct / train_total
    val_loss /= len(val_loader)
    val_acc = 100 * val_correct / val_total
    
    # Store history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['epochs'].append(epoch + 1)
    
    # Print progress
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    
    # Learning rate schedule
    scheduler.step(val_loss)
    
    # Early stopping and checkpoint (section 8)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        print(f"  → Best validation loss: {val_loss:.4f} ✓")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n⚠️  Early stopping at epoch {epoch+1} (patience exceeded)")
            break

elapsed = datetime.now() - start_time
print(f"\n{'='*70}")
print(f"Training Complete | {elapsed}")
print(f"{'='*70}")


## Section 7: Validate Model During Fine-Tuning

Visualize training progress and validation metrics.


In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(history['epochs'], history['train_loss'], 'o-', label='Train Loss', linewidth=2)
axes[0].plot(history['epochs'], history['val_loss'], 's-', label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(history['epochs'], history['train_acc'], 'o-', label='Train Acc', linewidth=2)
axes[1].plot(history['epochs'], history['val_acc'], 's-', label='Val Acc', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training & Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Training curves saved to {OUTPUT_DIR / 'training_curves.png'}")

# Summary table
print(f"\nTraining Summary:")
print(f"{'Epoch':<8} {'Train Loss':<12} {'Train Acc':<12} {'Val Loss':<12} {'Val Acc':<12}")
print("-" * 56)
for i, epoch in enumerate(history['epochs']):
    print(f"{epoch:<8} {history['train_loss'][i]:<12.4f} {history['train_acc'][i]:<12.2f} "
          f"{history['val_loss'][i]:<12.4f} {history['val_acc'][i]:<12.2f}")


## Section 8: Save Updated Checkpoint After Each Refinement Step

Persist the model weights and training metadata.


In [ ]:
# Save updated checkpoint
checkpoint = {
    'epoch': len(history['epochs']),
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'history': history,
    'best_val_loss': min(history['val_loss']),
    'best_val_acc': max(history['val_acc']),
    'timestamp': datetime.now().isoformat(),
    'hyperparams': {
        'learning_rate': LEARNING_RATE,
        'batch_size': BATCH_SIZE,
        'weight_decay': WEIGHT_DECAY,
        'num_classes': num_classes,
        'class_names': CANONICAL_CLASS_NAMES
    }
}

# Save to main checkpoint (overwrites previous)
output_checkpoint = 'best_indian_model.pth'
torch.save(checkpoint, output_checkpoint)
print(f"✓ Checkpoint saved: {output_checkpoint}")

# Also save a timestamped backup
backup_name = f"checkpoints/indian_model_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pth"
Path(backup_name).parent.mkdir(exist_ok=True)
torch.save(checkpoint, backup_name)
print(f"✓ Backup saved: {backup_name}")

# Save history JSON for analysis
history_file = OUTPUT_DIR / 'training_history.json'
with open(history_file, 'w') as f:
    json.dump({
        'epochs': history['epochs'],
        'train_loss': history['train_loss'],
        'train_acc': history['train_acc'],
        'val_loss': history['val_loss'],
        'val_acc': history['val_acc'],
        'best_val_loss': checkpoint['best_val_loss'],
        'best_val_acc': checkpoint['best_val_acc'],
        'timestamp': checkpoint['timestamp']
    }, f, indent=2)
print(f"✓ History saved: {history_file}")

print(f"\nCheckpoint Summary:")
print(f"  Best Val Loss: {checkpoint['best_val_loss']:.4f}")
print(f"  Best Val Acc: {checkpoint['best_val_acc']:.2f}%")
print(f"  Epochs trained: {checkpoint['epoch']}")
print(f"  Timestamp: {checkpoint['timestamp']}")


## Section 9: Run Post-Update Sanity Inference

Test the updated model on sample images to confirm it works correctly.


In [ ]:
# Load checkpoint to verify it was saved correctly
print("Sanity Check: Loading saved checkpoint...")
loaded_checkpoint = torch.load(output_checkpoint, map_location=device)

# Verify structure
assert 'model_state_dict' in loaded_checkpoint, "Missing model_state_dict"
assert 'history' in loaded_checkpoint, "Missing history"
print("✓ Checkpoint structure valid")

# Create new model and load weights
sanity_model = models.wide_resnet50_2(pretrained=False)
in_features = sanity_model.fc.in_features
sanity_model.fc = nn.Sequential(
    nn.Linear(in_features, num_classes),
    nn.LogSoftmax(dim=1)
)
sanity_model.load_state_dict(loaded_checkpoint['model_state_dict'])
sanity_model = sanity_model.to(device)
sanity_model.eval()
print("✓ Model loaded successfully")

# Test inference on a few validation samples
print("\nSanity Inference on Validation Samples:")
print(f"{'Sample':<8} {'True Label':<25} {'Predicted':<25} {'Confidence':<12}")
print("-" * 70)

with torch.no_grad():
    for i, (images, labels) in enumerate(val_loader):
        if i >= 1:  # Just one batch
            break
        images, labels = images.to(device), labels.to(device)
        outputs = sanity_model(images)
        probabilities = torch.exp(outputs)  # Convert LogSoftmax back to probabilities
        confidences, predicted = probabilities.max(1)
        
        for j in range(min(5, len(images))):  # Show first 5
            true_label = CANONICAL_CLASS_NAMES[labels[j].item()]
            pred_label = CANONICAL_CLASS_NAMES[predicted[j].item()]
            confidence = confidences[j].item()
            match = "✓" if labels[j].item() == predicted[j].item() else "✗"
            print(f"{match} {j+1:<6} {true_label:<25} {pred_label:<25} {confidence:<12.4f}")

print("\n" + "="*70)
print("✓ Sanity check complete! Model is ready for evaluation.")
print("="*70)

print(f"\n📊 Next Steps:")
print(f"1. Evaluate full validation set accuracy:")
print(f"   → Run per-class metrics and confusion matrix in next cell")
print(f"2. Test on Indian test images:")
print(f"   → python scripts/inference.py --image <path> --heatmap --model {output_checkpoint}")
print(f"3. Continue annotation and fine-tuning:")
print(f"   → Repeat cell annotation for more images, run this notebook again")


## Bonus: Per-Class Metrics & Confusion Matrix

Evaluate detailed performance metrics for each class.


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Evaluate on full validation set
all_preds = []
all_labels = []

sanity_model.eval()
with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Evaluating val set"):
        images, labels = images.to(device), labels.to(device)
        outputs = sanity_model(images)
        _, predicted = outputs.max(1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Per-class metrics
print("\nPer-Class Performance:")
print("="*70)
for class_idx, class_name in enumerate(CANONICAL_CLASS_NAMES):
    mask = all_labels == class_idx
    if mask.sum() > 0:
        class_acc = (all_preds[mask] == all_labels[mask]).mean() * 100
        n_samples = mask.sum()
        print(f"{class_name:<25} Accuracy: {class_acc:6.2f}% ({n_samples:3d} samples)")

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(10, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=CANONICAL_CLASS_NAMES,
            yticklabels=CANONICAL_CLASS_NAMES,
            cbar_kws={'label': 'Count'}, ax=ax)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title('Confusion Matrix - Validation Set')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Confusion matrix saved to {OUTPUT_DIR / 'confusion_matrix.png'}")

# Classification report
print("\n" + "="*70)
print("Classification Report:")
print("="*70)
print(classification_report(all_labels, all_preds, target_names=CANONICAL_CLASS_NAMES))


## Workflow Guide: Iterative Fine-Tuning

Use this notebook in a loop:

1. **Annotate cells** (outside notebook):
   ```bash
   python scripts/annotate_heatmap_cells.py --image data/IND_TESTING/COMPREHENSIVE_TEST/Forest/image.tif
   python scripts/annotate_cells_interactive.py --annotations image_annotations.json --only-incorrect
   python scripts/extract_corrected_cells.py --annotations image_annotations.json --image data/IND_TESTING/COMPREHENSIVE_TEST/Forest/image.tif
   ```

2. **Repeat for 3-10 images** — accumulate labeled cells in `data/cell_training/labeled_cells/`

3. **Run this notebook**:
   - Checkpoint path defaults to `best_eurosat_model.pth` for first run, then `best_indian_model.pth` for subsequent runs
   - Modify `CHECKPOINT_PATH` in Section 2 if needed
   - Execute all cells to train and save updated weights

4. **Evaluate results**:
   - Check training curves (Section 7)
   - Review per-class metrics (Bonus section)
   - Test on full heatmap: `python scripts/inference.py --image <path> --heatmap --model best_indian_model.pth`

5. **Next iteration**:
   - Label more cells (Round 2, 3, etc.)
   - Re-run notebook with accumulated data
   - Expected trajectory: 100 cells → 60%, 300 cells → 75%, 500 cells → 85%, 700+ cells → 90%

### Key Parameters to Modify

- **`CHECKPOINT_PATH`**: Which .pth file to load (changes each round)
- **`LEARNING_RATE`**: 1e-5 is safe; lower if memory limited, higher if loss plateaus
- **`BATCH_SIZE`**: 32 works; reduce to 16 if low on VRAM
- **`EPOCHS`**: 15 default; increase to 20 if not fully converged
- **`PATIENCE`**: 5 epochs; how many epochs without improvement before stopping
